In [1]:
!python -m pip install fastavro

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 40.0 MB/s eta 0:00:00a 0:00:01


In [6]:
import shutil
from pathlib import Path
from typing import Tuple, Dict, Any, List
from fastavro import reader as avro_reader, writer as avro_writer, parse_schema
ENCODING = "utf-8"

def crc_sidecar(path: Path) -> Path:
    """
    Hadoop local filesystem checksum sidecar:
      file:  metadata.json
      crc:   .metadata.json.crc
    """
    return path.with_name(f".{path.name}.crc")


def delete_crc_for(path: Path, verbose: bool = True) -> None:
    crc = crc_sidecar(path)
    if crc.exists():
        crc.unlink()
        if verbose:
            print(f"DELETED-CRC: {crc}")


def delete_crc_tree(root: Path, verbose: bool = True) -> int:
    count = 0
    for crc in root.rglob(".*.crc"):
        crc.unlink()
        count += 1
        if verbose:
            print(f"DELETED-CRC: {crc}")
    return count

def header_metadata(rdr) -> dict:
    """Pass through Iceberg's Avro file-level key/value metadata, minus reserved keys."""
    try:
        raw = (getattr(rdr, "_header", {}) or {}).get("meta", {}) or {}
    except Exception:
        return {}
    out = {}
    for k, v in raw.items():
        key = k.decode() if isinstance(k, (bytes, bytearray)) else k
        if key in ("avro.schema", "avro.codec"):   # handled separately by fastavro
            continue
        out[key] = v.decode() if isinstance(v, (bytes, bytearray)) else v
    return out
    

def backup_once(path: Path):
    bak = path.with_suffix(path.suffix + ".bak")
    if not bak.exists():
        shutil.copy2(path, bak)
    return bak

def detect_codec(rdr) -> str | None:
    try:
        meta = getattr(rdr, "_header", {}).get("meta", {})
        val = meta.get(b"avro.codec") or meta.get("avro.codec")
        return val.decode() if isinstance(val, (bytes, bytearray)) else val
    except Exception:
        return None


def replace_in_record(rec: Dict[str, Any], old_prefix: str, new_prefix: str) -> int:
    """Replace file_path / manifest_path if they contain the old prefix."""
    changes = 0
    for key in ("file_path", "manifest_path"):
        if key in rec and isinstance(rec[key], str) and old_prefix in rec[key]:
            rec[key] = rec[key].replace(old_prefix, new_prefix)
            changes += 1
    return changes


def walk_replace(obj: Any, old_prefix: str, new_prefix: str) -> tuple[Any, int]:
    """Recursively replace old_prefix in any string value; return (new_obj, num_changes)."""
    changes = 0
    if isinstance(obj, dict):
        out = {}
        for k, v in obj.items():
            nv, c = walk_replace(v, old_prefix, new_prefix)
            out[k] = nv
            changes += c
        return out, changes
    elif isinstance(obj, list):
        out = []
        for v in obj:
            nv, c = walk_replace(v, old_prefix, new_prefix)
            out.append(nv)
            changes += c
        return out, changes
    elif isinstance(obj, str):
        if old_prefix in obj:
            return obj.replace(old_prefix, new_prefix), 1
        return obj, 0
    else:
        return obj, 0
        

def rewrite_avro_file(path: Path, old_prefix: str, new_prefix: str, apply: bool, verbose: bool) -> Tuple[int, int]:
    """Rewrite any string fields containing old_prefix. Returns (records_changed, fields_changed)."""
    records: List[Dict[str, Any]] = []
    records_changed = 0
    fields_changed = 0

    with path.open("rb") as fh:
        rdr = avro_reader(fh)
        schema = rdr.writer_schema
        codec = detect_codec(rdr)
        meta = header_metadata(rdr)
        for rec in rdr:
            rec_new, c = walk_replace(rec, old_prefix, new_prefix)
            if c:
                records_changed += 1
                fields_changed += c
            records.append(rec_new)

    if fields_changed and apply:
        backup_once(path)
        tmp = path.with_suffix(path.suffix + ".tmp")
        try:
            with tmp.open("wb") as out:
                if codec:
                    avro_writer(out, parse_schema(schema), records, codec=codec,metadata=(meta or None))   
                else:
                    avro_writer(out, parse_schema(schema), records, metadata=(meta or None))
            tmp.replace(path)
            delete_crc_for(path, verbose=verbose)
            if verbose:
                print(f"UPDATED: {path} (+{fields_changed} changes across {records_changed} records) [codec={codec or 'null'}]")
        finally:
            if tmp.exists():
                try:
                    tmp.unlink()
                except Exception:
                    pass
    elif fields_changed and not apply and verbose:
        print(f"WOULD-UPDATE: {path} (+{fields_changed} changes across {records_changed} records)")

    return records_changed, fields_changed


def run_update_avro(root: Path, old_prefix: str, new_prefix: str, apply: bool, verbose: bool = True):
    total_files = 0
    changed_files = 0
    total_rec = 0
    total_fields = 0

    for av in root.rglob("metadata/*.avro"):
        total_files += 1
        try:
            rec_chg, fld_chg = rewrite_avro_file(av, old_prefix, new_prefix, apply, verbose)
            if fld_chg:
                changed_files += 1
                total_rec += rec_chg
                total_fields += fld_chg
        except Exception as e:
            print(f"# WARN: failed to process {av}: {e}")

    print("\nSummary:")
    print(f"  AVRO files scanned : {total_files}")
    print(f"  Files with changes : {changed_files}")
    print(f"  Records changed    : {total_rec}")
    print(f"  Field updates      : {total_fields}")
    print("  Mode               :", "APPLY (writes performed, .bak created)" if apply else "DRY-RUN (no writes)")


In [8]:
ROOT = Path(r"/mnt/iceberg-ldbc-snb-1tb/warehouse/ldbc_cnb_sf1000.db") 
OLD_PREFIX = r"/mnt/iceberg/warehouse/ldbc_cnb_sf1000.db/"
NEW_PREFIX = "/mnt/iceberg/warehouse/ldbc_snb_sf1000.db/"      

APPLY = False
VERBOSE = True

run_update_avro(ROOT, OLD_PREFIX, NEW_PREFIX, APPLY, verbose=VERBOSE)

UPDATED: /mnt/iceberg-ldbc-snb-1tb/warehouse/ldbc_cnb_sf1000.db/forum_has_member_person/metadata/snap-5778376579802736353-1-d7ec96da-dff1-4834-94b0-8c6b509dc4dd.avro (+1 changes across 1 records) [codec=deflate]
UPDATED: /mnt/iceberg-ldbc-snb-1tb/warehouse/ldbc_cnb_sf1000.db/forum_has_member_person/metadata/d7ec96da-dff1-4834-94b0-8c6b509dc4dd-m0.avro (+2687 changes across 2687 records) [codec=deflate]
UPDATED: /mnt/iceberg-ldbc-snb-1tb/warehouse/ldbc_cnb_sf1000.db/person_likes_comment/metadata/snap-1017963673949537918-1-caea231d-8f30-404e-a3a8-c3cb26146969.avro (+1 changes across 1 records) [codec=deflate]
UPDATED: /mnt/iceberg-ldbc-snb-1tb/warehouse/ldbc_cnb_sf1000.db/person_likes_comment/metadata/caea231d-8f30-404e-a3a8-c3cb26146969-m0.avro (+2015 changes across 2015 records) [codec=deflate]
UPDATED: /mnt/iceberg-ldbc-snb-1tb/warehouse/ldbc_cnb_sf1000.db/forum_has_tag_tag/metadata/snap-8978759131098150570-1-892f2e77-5394-499f-8191-d1e2ede42a88.avro (+1 changes across 1 records) [cod